In [ ]:
import json
import os
import re
from pathlib import Path
from typing import Dict, Any, List

def build_master_logo_table(root: Path) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for json_path in sorted(root.rglob("results.json")):
        # Restrict to LOGO only
        if f"{os.sep}logo{os.sep}" not in str(json_path):
            continue

        meta = parse_run_metadata(json_path)
        results = load_logo_results(json_path)

        for cow_id_str, metrics in results.items():
            # Skip any non-cow keys
            if not str(cow_id_str).isdigit():
                continue

            row = dict(meta)
            row["cow_id"] = int(cow_id_str)

            if isinstance(metrics, dict):
                for k, v in metrics.items():
                    if isinstance(v, (int, float, str, bool)) or v is None:
                        row[k] = v
            else:
                row["metrics_raw"] = str(metrics)

            # Convenience columns
            support_lying = row.get("support_lying")
            support_standing = row.get("support_standing")

            # Sometimes these are floats in your JSON; treat as numeric
            support_lying = float(support_lying) if support_lying is not None else 0.0
            support_standing = float(support_standing) if support_standing is not None else 0.0

            row["support_total"] = support_lying + support_standing
            row["lying_ratio_obs"] = (
                support_lying / (row["support_total"] + EPS)
                if row["support_total"] > 0
                else None
            )

            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            f"No LOGO results.json files found under {root}. "
            f"Check ROOT and folder structure."
        )

    # Put key columns first if they exist
    preferred = [
        "cow_id",
        "feature_engineering",
        "feature_set",
        "variant_name",
        "exp_id",
        "eval_kind",
        "macro_f1",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1_lying",
        "f1_standing",
        "support_lying",
        "support_standing",
        "support_total",
        "lying_ratio_obs",
        "use_markov",
        "path",
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    df = df[cols]

    return df


master_logo = build_master_logo_table(ROOT)
master_logo.shape


In [2]:
"""
Cow-level distribution shift vs. LOGO performance
------------------------------------------------
Assumes you already have:
  - df: pandas DataFrame with columns:
      * 'animal_id'
      * IMU_Tick_Count_XXmG columns (e.g., 40,80,...,240)
  - logo_scores: cow-level LOGO results DataFrame OR dict (optional)
      * must contain 'animal_id' and 'macro_f1' at minimum

This script:
  1) finds IMU tick columns
  2) aggregates per-cow feature means (and optional variances)
  3) computes distance-to-population using:
       - Mahalanobis distance (covariance-aware) OR fallback
       - z-score distance (robust and simple)
  4) plots distance vs macro-F1 in a thesis-friendly style (matplotlib only)
"""

from __future__ import annotations

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/luka/priv_projects/moomotion/data/CWB_2024.csv")

# -----------------------------
# 0) User inputs (edit these)
# -----------------------------
ANIMAL_COL = "animal_id"
# If you have LOGO macro-F1 per cow, put it here:
# Option A: DataFrame with ['animal_id','macro_f1']
# Option B: dict {animal_id: macro_f1}
logo_scores = None  # <- set this

# Distance settings
USE_ROBUST_Z = True          # robust z using median/MAD
USE_MAHALANOBIS = True       # try Mahalanobis; will fallback safely
COV_SHRINKAGE = 1e-3         # stabilizes covariance inversion

# Plot settings
FIGSIZE = (7.2, 4.5)         # thesis-friendly
ALPHA = 0.85
S = 40

# -----------------------------
# 1) Helpers
# -----------------------------
def find_imu_tick_cols(columns: list[str]) -> list[str]:
    pat = re.compile(r"^IMU_Tick_Count_(\d+)mG$")
    cols = []
    for c in columns:
        m = pat.match(c)
        if m:
            cols.append(c)
    # sort by threshold value
    cols.sort(key=lambda x: int(pat.match(x).group(1)))
    return cols

def robust_center_scale(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns (center, scale) using median and MAD.
    scale uses 1.4826*MAD to approximate std under normality.
    """
    center = np.nanmedian(X, axis=0)
    mad = np.nanmedian(np.abs(X - center), axis=0)
    scale = 1.4826 * mad
    # avoid divide-by-zero
    scale = np.where(scale == 0, 1.0, scale)
    return center, scale

def mean_std_center_scale(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    center = np.nanmean(X, axis=0)
    scale = np.nanstd(X, axis=0, ddof=1)
    scale = np.where((scale == 0) | np.isnan(scale), 1.0, scale)
    return center, scale

def mahalanobis_distance(X: np.ndarray, center: np.ndarray, cov: np.ndarray) -> np.ndarray:
    """
    Computes Mahalanobis distance of each row of X to 'center' with covariance 'cov'.
    """
    Xc = X - center
    # add ridge to covariance for numerical stability
    cov_reg = cov + np.eye(cov.shape[0]) * COV_SHRINKAGE
    inv_cov = np.linalg.inv(cov_reg)
    # d^2 = x^T inv_cov x
    d2 = np.einsum("ij,jk,ik->i", Xc, inv_cov, Xc)
    d = np.sqrt(np.maximum(d2, 0))
    return d

def spearmanr_safe(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """
    Spearman correlation without SciPy. Returns (rho, approx_p).
    p-value is approximate (normal approximation). Good enough for an exploratory figure caption.
    If you have SciPy, replace with scipy.stats.spearmanr.
    """
    # rank with average ties
    xr = pd.Series(x).rank(method="average").to_numpy()
    yr = pd.Series(y).rank(method="average").to_numpy()
    xr = xr - xr.mean()
    yr = yr - yr.mean()
    denom = (np.sqrt((xr**2).sum()) * np.sqrt((yr**2).sum()))
    rho = float((xr * yr).sum() / denom) if denom > 0 else np.nan

    # approximate p-value using t approximation (rough)
    n = np.isfinite(x).sum()
    if n < 4 or not np.isfinite(rho):
        return rho, np.nan
    t = rho * np.sqrt((n - 2) / max(1e-12, (1 - rho**2)))
    # two-sided p from normal approx of t (rough)
    # Use erf approximation:
    from math import erf, sqrt
    p = 2 * (1 - 0.5 * (1 + erf(abs(t) / sqrt(2))))
    return rho, p

# -----------------------------
# 2) Extract features and clean
# -----------------------------
imu_cols = find_imu_tick_cols(list(df.columns))
if not imu_cols:
    raise ValueError("No IMU tick columns found. Expected columns like 'IMU_Tick_Count_80mG'.")

needed = [ANIMAL_COL] + imu_cols
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_feat = df[needed].copy()

# Ensure numeric
for c in imu_cols:
    df_feat[c] = pd.to_numeric(df_feat[c], errors="coerce")

# Drop rows with no animal_id
df_feat = df_feat.dropna(subset=[ANIMAL_COL])

# Optional: drop rows where all IMU are NaN
df_feat = df_feat.dropna(subset=imu_cols, how="all")

# -----------------------------
# 3) Aggregate per cow
# -----------------------------
cow_agg = (
    df_feat.groupby(ANIMAL_COL)[imu_cols]
    .agg(["mean", "std", "count"])
)

# Flatten multiindex columns
cow_agg.columns = ["_".join(col).strip() for col in cow_agg.columns.to_flat_index()]
cow_agg = cow_agg.reset_index()

# Keep a simple support measure: min count across IMU columns
count_cols = [f"{c}_count" for c in imu_cols]
cow_agg["support_total"] = cow_agg[count_cols].min(axis=1)

# Build matrix of per-cow means (this is what we measure distance on)
mean_cols = [f"{c}_mean" for c in imu_cols]
X = cow_agg[mean_cols].to_numpy(dtype=float)

# -----------------------------
# 4) Compute distance-to-population
# -----------------------------
# Center/scale for z-distance
if USE_ROBUST_Z:
    center_z, scale_z = robust_center_scale(X)
else:
    center_z, scale_z = mean_std_center_scale(X)

Z = (X - center_z) / scale_z
cow_agg["z_dist"] = np.sqrt(np.nanmean(Z**2, axis=1))  # RMS z distance

# Mahalanobis distance (optional)
if USE_MAHALANOBIS:
    # covariance of per-cow means
    # Use nan-safe covariance: fill remaining NaNs with column center
    X_cov = X.copy()
    nan_mask = np.isnan(X_cov)
    if nan_mask.any():
        X_cov[nan_mask] = np.take(center_z, np.where(nan_mask)[1])
    cov = np.cov(X_cov, rowvar=False)
    try:
        cow_agg["mahal_dist"] = mahalanobis_distance(X_cov, center=np.nanmean(X_cov, axis=0), cov=cov)
    except np.linalg.LinAlgError:
        cow_agg["mahal_dist"] = np.nan

# Choose distance to plot (prefer mahal if it worked)
dist_col = "mahal_dist" if (USE_MAHALANOBIS and cow_agg["mahal_dist"].notna().any()) else "z_dist"

# -----------------------------
# 5) Merge in LOGO macro-F1 (if provided)
# -----------------------------
if logo_scores is None:
    # You can still inspect cow_agg and later merge
    print("logo_scores is None: plotting distance only. Provide per-cow macro_f1 to plot distance vs performance.")
else:
    if isinstance(logo_scores, dict):
        score_df = pd.DataFrame({ANIMAL_COL: list(logo_scores.keys()), "macro_f1": list(logo_scores.values())})
    elif isinstance(logo_scores, pd.DataFrame):
        if ANIMAL_COL not in logo_scores.columns or "macro_f1" not in logo_scores.columns:
            raise ValueError("logo_scores DataFrame must contain columns ['animal_id','macro_f1'].")
        score_df = logo_scores[[ANIMAL_COL, "macro_f1"]].copy()
    else:
        raise TypeError("logo_scores must be a dict or a pandas DataFrame.")

    plot_df = cow_agg.merge(score_df, on=ANIMAL_COL, how="inner")
    plot_df = plot_df.dropna(subset=[dist_col, "macro_f1"])

    # -----------------------------
    # 6) Plot (matplotlib only)
    # -----------------------------
    plt.figure(figsize=FIGSIZE)
    plt.scatter(
        plot_df[dist_col].to_numpy(),
        plot_df["macro_f1"].to_numpy(),
        s=S,
        alpha=ALPHA,
    )
    plt.xlabel("Distance to population feature distribution")
    plt.ylabel("Cow-level macro-F1 (LOGO)")
    plt.ylim(0.0, 1.02)

    # Optional: annotate a few worst performers
    worst = plot_df.nsmallest(5, "macro_f1")
    for _, r in worst.iterrows():
        plt.annotate(
            str(int(r[ANIMAL_COL])) if pd.api.types.is_numeric_dtype(plot_df[ANIMAL_COL]) else str(r[ANIMAL_COL]),
            (r[dist_col], r["macro_f1"]),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=8,
        )

    # correlation (Spearman)
    rho, p = spearmanr_safe(plot_df[dist_col].to_numpy(), plot_df["macro_f1"].to_numpy())
    plt.title(f"LOGO performance vs distribution shift (Spearman ρ={rho:.2f})")

    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

# -----------------------------
# 7) Export a thesis table (optional)
# -----------------------------
# This gives you a cow-level table you can use in appendix or debugging.
out_cols = [ANIMAL_COL, "support_total", "z_dist"]
if "mahal_dist" in cow_agg.columns:
    out_cols.append("mahal_dist")

cow_distance_table = cow_agg[out_cols].sort_values(by=(dist_col if dist_col in cow_agg.columns else "z_dist"), ascending=False)
print(cow_distance_table.head(10))


logo_scores is None: plotting distance only. Provide per-cow macro_f1 to plot distance vs performance.
    animal_id  support_total    z_dist  mahal_dist
14      15576              4  1.255822    6.651701
94      99750              3  1.801832    6.482185
2        2460            149  1.593662    4.899005
85      99730              6  1.888199    4.861677
73      99691              9  1.670559    4.322276
91      99744            143  3.387624    4.175511
20      21854            541  3.387624    4.175511
11      15560              3  3.387624    4.175511
43      37444            143  3.387624    4.175511
68      75006              4  1.721510    4.074714
